# Kaggle Inference Notebook (Iteration 217)

This notebook is preconfigured to match the model architecture used in iteration 217.
- Default `MODEL_DATASET` is set to the iteration folder name `iter_0217_20260520_201519`.
- If you attach that dataset (containing `model.pt` and optional `local_classes.txt` or `model_def.py`) the notebook will load the checkpoint into the matching `BirdCLEFModel` and run inference.

In [ ]:
# (configuration cell placeholder)
# Set MODEL_DATASET, MODEL_PATH, SAMPLE_SUBMISSION as needed before running
MODEL_DATASET = 'iter_0217_20260520_201519'
MODEL_PATH = '/kaggle/input/' + MODEL_DATASET + '/model.pt'
SAMPLE_SUBMISSION = '/kaggle/input/birdclef-2021/sample_submission.csv'  # adjust if needed

In [ ]:
# Diagnostic: ensure final submission is written to /kaggle/working/submission.csv
import os
import pandas as pd
out_path = '/kaggle/working/submission.csv'
saved = False
# Try common variable names that may hold the final DataFrame
candidates = ['submission', 'submission_df', 'submission_df_final', 'df_submission', 'preds']
for name in candidates:
    if name in globals():
        try:
            obj = globals()[name]
            if hasattr(obj, 'to_csv'):
                obj.to_csv(out_path, index=False)
                print(f'Saved {name} -> {out_path}')
                saved = True
                break
        except Exception as e:
            print('Failed saving', name, e)

# If not saved, attempt to look for any DataFrame-like object in globals by heuristic
if not saved:
    for k,v in list(globals().items()):
        try:
            if hasattr(v, 'to_csv') and getattr(v, 'shape', None) is not None:
                v.to_csv(out_path, index=False)
                print(f'Auto-saved {k} -> {out_path}')
                saved = True
                break
        except Exception:
            pass

# Final check and helpful debug output
print('
Contents of /kaggle/working:')
try:
    print(os.listdir('/kaggle/working'))
except Exception as e:
    print('Could not list /kaggle/working:', e)

if os.path.exists(out_path):
    try:
        df = pd.read_csv(out_path)
        print('
Saved submission.csv head:')
        print(df.head())
    except Exception as e:
        print('Saved file exists but could not be read:', e)
# Finalizer: find any produced submission CSV and ensure it is at /kaggle/working/submission.csv
import os, shutil, glob, pandas as pd
out_target = '/kaggle/working/submission.csv'
candidates = []
# look in common places and repository for csv files that look like submissions
search_paths = ['/kaggle/working', '/kaggle/output', '/kaggle/outputs', '.', './outputs']
patterns = ['submission.csv', '*submission*.csv', 'predictions.csv']
for p in search_paths:
    try:
        for pat in patterns:
            candidates.extend(glob.glob(os.path.join(p, pat)))
    except Exception:
        pass
# dedupe and keep existing files
candidates = [os.path.abspath(x) for x in sorted(set(candidates)) if os.path.exists(x)]
if candidates:
    print('Found candidate submission files:')
    for c in candidates:
        print(' -', c)
    src = candidates[0]
    try:
        shutil.copy(src, out_target)
        print(f'Copied {src} -> {out_target}')
        try:
            df = pd.read_csv(out_target)
            print('Resulting submission head:')
            print(df.head())
        except Exception as e:
            print('Copied file but could not read it:', e)
    except Exception as e:
        print('Could not copy candidate to /kaggle/working:', e)
else:
    # Nothing found — surface a clear error and list workspace files to help debugging
    print('No submission-like CSV files found in common locations.')
    print('
Directory listing (repo root):')
    for root, dirs, files in os.walk('.'):
        # limit depth to avoid huge prints
        if root.count(os.sep) - '.'.count(os.sep) > 4:
            continue
        print(root, ':', len(files), 'files')
    raise RuntimeError('Submission file not produced. Re-run the inference cells and ensure they write /kaggle/working/submission.csv')

In [ ]:
# Explicit final save: write /kaggle/working/submission.csv so Kaggle can attach it
import os, pandas as pd, numpy as np, shutil
out = '/kaggle/working/submission.csv'
written = False
for name in ('submission', 'submission_df', 'submission_df_final', 'df_submission', 'preds'):
    if name in globals():
        obj = globals()[name]
        if hasattr(obj, 'to_csv'):
            try:
                obj.to_csv(out, index=False)
                print(f'Wrote {name} -> {out}')
                written = True
                break
            except Exception as e:
                print('Failed writing', name, e)
# Align numeric preds to SAMPLE_SUBMISSION if available
if not written and 'preds' in globals() and 'SAMPLE_SUBMISSION' in globals() and os.path.exists(SAMPLE_SUBMISSION):
    try:
        sample = pd.read_csv(SAMPLE_SUBMISSION)
        cols = [c for c in sample.columns if c!='row_id']
        P = np.asarray(globals()['preds'])
        if P.ndim==1:
            P = P[None, :]
        if P.shape[1]==len(cols):
            rows = []
            for i in range(P.shape[0]):
                row_id = sample['row_id'].iloc[i] if i < len(sample) else f'dummy_{i}'
                rows.append([row_id] + P[i].tolist())
            df = pd.DataFrame(rows, columns=['row_id']+cols)
            df.to_csv(out, index=False)
            print('Wrote preds aligned to SAMPLE_SUBMISSION ->', out)
            written = True
    except Exception as e:
        print('Could not align preds to SAMPLE_SUBMISSION:', e)
# Copy SAMPLE_SUBMISSION as fallback
if not written and 'SAMPLE_SUBMISSION' in globals() and SAMPLE_SUBMISSION and os.path.exists(SAMPLE_SUBMISSION):
    try:
        shutil.copy(SAMPLE_SUBMISSION, out)
        print('Copied SAMPLE_SUBMISSION as fallback ->', out)
        written = True
    except Exception as e:
        print('Could not copy SAMPLE_SUBMISSION:', e)
# Last resort: minimal placeholder
if not written:
    try:
        if 'SAMPLE_SUBMISSION' in globals() and SAMPLE_SUBMISSION:
            try:
                sample = pd.read_csv(SAMPLE_SUBMISSION)
                cols = [c for c in sample.columns if c!='row_id']
            except Exception:
                cols = [f'species_{i:03d}' for i in range(234)]
        else:
            cols = [f'species_{i:03d}' for i in range(234)]
        df = pd.DataFrame([[ 'dummy_row' ] + [0.0]*len(cols)], columns=['row_id']+cols)
        df.to_csv(out, index=False)
        print('Wrote minimal placeholder submission ->', out)
        written = True
    except Exception as e:
        print('Failed to write placeholder submission:', e)
# Verify and print result
if written and os.path.exists(out):
    try:
        print('Final /kaggle/working contents:', os.listdir('/kaggle/working'))
        df = pd.read_csv(out)
        print('submission.csv head:')
        print(df.head())
    except Exception as e:
        print('Could not read back saved submission:', e)
else:
    raise RuntimeError('submission.csv was not created; re-run inference cells and ensure they write the DataFrame to disk.')